# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-8.18125403e-01  4.09624313e-01  7.07602654e-01 -4.71781068e-01
  -6.00911067e-01]
 [-5.06162678e-01 -1.59921798e-01 -5.31528304e-01 -1.54958413e-04
  -7.98551544e-02]
 [-4.35720734e-01 -2.21034143e-01  1.89347958e-01  5.50346353e-01
   1.46017519e-01]
 [-3.32089871e-01 -5.34941107e-01  3.66131145e-01 -4.89046495e-01
  -1.95161653e-02]
 [-5.91605122e-01  2.83161429e-03 -2.07889276e-02 -3.86875502e-01
   2.36247500e-01]
 [ 4.63574294e-01  1.97235909e-01 -8.83439479e-01 -3.42466545e-01
  -5.68418700e-01]
 [-8.61885720e-01 -2.39958631e-01  8.29015122e-01 -7.22029327e-01
  -6.30991866e-01]
 [-7.28111550e-01 -6.98856984e-01 -9.72726270e-02 -6.41501176e-01
   7.85598365e-01]
 [ 9.41502904e-01  5.36868894e-01 -3.95789164e-01 -2.75522217e-01
   7.48041890e-01]
 [-4.60022001e-01 -8.61810553e-01  6.69726574e-01 -9.91834449e-01
  -5.97383635e-01]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a2', 'a1', 'a1', 'a1', 'a1', 'a2', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 0, 0, 0, 0, 0, 1, 1, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:00<00:30,  1.08it/s]

SVI:   3%|▎         | 1/34 [00:00<00:30,  1.08it/s, loss=2216.8005]

SVI:   6%|▌         | 2/34 [00:00<00:29,  1.08it/s, loss=2390.4583]

SVI:   9%|▉         | 3/34 [00:00<00:28,  1.08it/s, loss=1850.5924]

...

SVI:  84%|████████▍ | 21/25 [00:01<00:04,  1.01s/it, loss=1934.6333]

SVI:  88%|████████▊ | 22/25 [00:01<00:03,  1.01s/it, loss=2226.4929]

SVI:  92%|█████████▏| 23/25 [00:01<00:02,  1.01s/it, loss=2342.4634]

SVI:  96%|█████████▌| 24/25 [00:01<00:01,  1.01s/it, loss=2135.0320]

SVI: 100%|██████████| 25/25 [00:01<00:00,  1.01s/it, loss=2358.7698]